# Lab 03 
## Made by: Santiago Villa Rodríguez
## On: February 18th, 2026

In [1]:
from spark_utils import SparkUtils

su = SparkUtils("spark://spark-master:7077", "Lab03 - Data Cleaning and Transformation Pipeline")
su._spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 00:34:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Create the Quick Commerce Schema

In [5]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,Product_Category,
Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, LongType

columns_types = [
    ("Order_ID", "long"),
    ("Company", "string"),
    ("City", "string"),
    ("Customer_Age", "int"),
    ("Order_Value", "float"),
    ("Delivery_Time_Min", "float"),
    ("Distance_Km", "float"),
    ("Items_Count", "float"),
    ("Product_Category", "string"),
    ("Payment_Method", "string"),
    ("Customer_Rating", "float"),
    ("Discount_Applied", "float"),
    ("Delivery_Partner_Rating", "float")
]

qcommerce_schema = StructType([
    StructField("Order_ID", LongType(), True),
    StructField("Company", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Customer_Age", IntegerType(), True),
    StructField("Order_Value", FloatType(), True),
    StructField("Delivery_Time_Min", FloatType(), True),
    StructField("Distance_Km", FloatType(), True),
    StructField("Items_Count", FloatType(), True),
    StructField("Product_Category", StringType(), True),
    StructField("Payment_Method", StringType(), True),
    StructField("Customer_Rating", FloatType(), True),
    StructField("Discount_Applied", FloatType(), True),
    StructField("Delivery_Partner_Rating", FloatType(), True)
])

# qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
    .read \
    .option("header", "true") \
    .schema(qcommerce_schema) \
    .csv("/opt/spark/work-dir/data/qcommerce/")

qcommerce_df.printSchema()
print("Number of rows", qcommerce_df.count())
qcommerce_df.show(5)


root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

Number of rows 1000000
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+-------

### Counting Nulls

In [7]:
from pyspark.sql.functions import when, count, isnull

qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()


+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



### Removing Nulls

In [8]:
qcommerce_wo_nulls_df = qcommerce_df.dropna()
print(f"Size of data frame after removing nulls: {qcommerce_wo_nulls_df.count()}")

Size of data frame after removing nulls: 780992


### Filling Nulls

In [9]:
qcommerce_wo_nulls_fillna_df = qcommerce_df.fillna({
    "City": "Unknown",
    "Items_Count": 0.0,
    "Customer_Rating": 0.0,
    "Delivery_Partner_Rating": 0.0
})
print(f"Size of data frame after filling nulls: {qcommerce_wo_nulls_fillna_df.count()}")

Size of data frame after filling nulls: 1000000


### Creating a new column

In [10]:
from pyspark.sql.functions import lit

qcommerce_wo_nulls_fillna_df = qcommerce_wo_nulls_fillna_df.withColumn("Code_Country", lit("IN"))
qcommerce_wo_nulls_fillna_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

### Creating TAX column

In [11]:
qcommerce_tax_df = qcommerce_wo_nulls_fillna_df.withColumn("Tax", qcommerce_wo_nulls_fillna_df["Order_Value"] * 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|             Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

## Task 1 - Creating Delivery_SLA

In [17]:
from pyspark.sql.functions import col

qcommerce_df_task1 = qcommerce_tax_df.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST") \
                                                                .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON TIME") \
                                                                .when(col("Delivery_Time_Min") > 20, "LATE")) \
                                                                .filter(col("Delivery_SLA") == "LATE") \
                                                                .orderBy(col("Delivery_Time_Min"), ascending=False)

qcommerce_df_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()


+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

## Task 2 - Creating Price_Bucket

In [22]:
from pyspark.sql.functions import avg

qcommerce_effective_order_value = qcommerce_df_task1.withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied")))

qcommerce_df_task2 = qcommerce_effective_order_value.withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW") \
                                                                                .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") < 500), "MEDIUM") \
                                                                                .when(col("Effective_Order_Value") >= 500, "HIGH")) \
                                                                                .groupBy("Price_Bucket").agg(
                                                                                    count("*").alias("Count_Price_Bucket"),
                                                                                    avg("Effective_Order_Value").alias("Avg_Effective_Order")
                                                                                ) \
                                                                                .orderBy(col("Avg_Effective_Order"), ascending=False)   

qcommerce_df_task2.show()       

+------------+------------------+-------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Order|
+------------+------------------+-------------------+
|        HIGH|             56013|   707.219837039914|
|      MEDIUM|             59824| 353.49134017350553|
|         LOW|            143811|  25.36497016341368|
+------------+------------------+-------------------+



## Task 3

In [26]:
qcommerce_df_task3 = qcommerce_df_task1.withColumn("Age_Group", when(col("Customer_Age") < 25.0, "YOUNG") \
                                                                .when((col("Customer_Age") >= 25.0) & (col("Customer_Age") < 44.0), "ADULT") \
                                                                .when(col("Customer_Age") >= 44.0, "SENIOR")) \
                                                                .filter((col("Customer_Age").isNotNull()) & (col("Customer_Age") != 0.0) & (col("Customer_Age") != 100.0)) \
                                                                .groupBy("Age_Group", "Product_Category").agg(
                                                                    count("*").alias("Orders"),
                                                                    avg("Order_Value").alias("Avg_Order_Value")
                                                                ) \
                                                                .orderBy(col("Age_Group"), col("Orders"), ascending=[True, False])

qcommerce_df_task3.show()

+---------+-------------------+------+------------------+
|Age_Group|   Product_Category|Orders|   Avg_Order_Value|
+---------+-------------------+------+------------------+
|    ADULT|      Personal Care| 17041| 495.5667582791104|
|    ADULT|          Household| 16899|  497.352523035083|
|    ADULT|              Dairy| 16874| 502.2252479181112|
|    ADULT|          Beverages| 16801| 496.4155118538222|
|    ADULT|Fruits & Vegetables| 16788|  493.813331524247|
|    ADULT|             Snacks| 16786| 499.6387309673559|
|    ADULT|          Groceries| 16672| 491.6697944457609|
|   SENIOR|          Groceries| 14313|499.66015971475684|
|   SENIOR|             Snacks| 14164| 502.8156542868239|
|   SENIOR|      Personal Care| 14147|501.26034197136903|
|   SENIOR|              Dairy| 14082| 499.1617355167891|
|   SENIOR|          Household| 14031|489.94910961284614|
|   SENIOR|          Beverages| 13967| 492.9890682345822|
|   SENIOR|Fruits & Vegetables| 13838| 498.8159517461052|
|    YOUNG|   

In [27]:
su.stop()